# Clean MF-GP Workflow (No resum helpers)

This notebook trains/evaluates a clean two-level autoregressive multi-fidelity GP
from CNP outputs and saves comparable diagnostics/plots to `path_out_mfgp`.

In [ ]:
from pathlib import Path
import importlib
import json
import pandas as pd
from IPython.display import Image, display

import mfgp_clean_pipeline as mfgp_clean_pipeline
mfgp_clean_pipeline = importlib.reload(mfgp_clean_pipeline)
run_clean_mfgp = mfgp_clean_pipeline.run_clean_mfgp
load_runtime_config = mfgp_clean_pipeline.load_runtime_config


In [ ]:
# Update if needed
CONFIG_PATH = Path('../xenon/settings2.yaml').resolve()
runtime = load_runtime_config(CONFIG_PATH)

print('config      :', runtime.config_path)
print('version     :', runtime.version)
print('path_out_cnp:', runtime.out_dir_cnp)
print('path_out_mfgp:', runtime.out_dir_mfgp)


In [ ]:
# Use separate files for MF-GP fit vs validation-style extra plots
# Set to None to auto-discover under runtime.out_dir_cnp
TRAIN_CNP_CSV = None
VALIDATION_CNP_CSV = None

ITERATION = 0
LF_FIDELITY = 0
HF_FIDELITY = 1
GRID_POINTS = 120
PREDICT_CHUNK_SIZE = 20000
VERBOSE = True

def _pick_train_csv(out_dir, version):
    pats = sorted(out_dir.glob(f'cnp_{version}_output_*epochs.csv'))
    pats = [p for p in pats if 'validation' not in p.name and 'per_signal' not in p.name]
    pats.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return pats[0] if pats else None

def _pick_val_csv(out_dir, version):
    pats = sorted(out_dir.glob(f'cnp_{version}_output_validation_*epochs.csv'))
    pats = [p for p in pats if 'per_signal' not in p.name]
    pats.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return pats[0] if pats else None

train_csv = Path(TRAIN_CNP_CSV).expanduser().resolve() if TRAIN_CNP_CSV else _pick_train_csv(runtime.out_dir_cnp, runtime.version)
val_csv = Path(VALIDATION_CNP_CSV).expanduser().resolve() if VALIDATION_CNP_CSV else _pick_val_csv(runtime.out_dir_cnp, runtime.version)

print('out_dir_cnp :', runtime.out_dir_cnp)
print('train_csv   :', train_csv, '| exists:', bool(train_csv and train_csv.exists()))
print('val_csv     :', val_csv, '| exists:', bool(val_csv and val_csv.exists()))
print('available   :')
for p in sorted(runtime.out_dir_cnp.glob(f'cnp_{runtime.version}_*epochs.csv')):
    print('  -', p.name)

result = run_clean_mfgp(
    config_path=CONFIG_PATH,
    cnp_csv=str(train_csv) if train_csv else None,
    validation_csv=str(val_csv) if val_csv else None,
    iteration=ITERATION,
    lf_fidelity=LF_FIDELITY,
    hf_fidelity=HF_FIDELITY,
    grid_points_per_axis=GRID_POINTS,
    random_state=42,
    prefer_validation_csv=False,
    predict_chunk_size=PREDICT_CHUNK_SIZE,
    verbose=VERBOSE,
)
result


In [ ]:
metrics = json.loads(Path(result.metrics_json).read_text())
metrics


In [ ]:
df_pred = pd.read_csv(result.prediction_csv)
print(df_pred.shape)
df_pred.head()


In [ ]:
if result.data_plot:
    display(Image(filename=str(result.data_plot)))
if result.parity_plot:
    display(Image(filename=str(result.parity_plot)))
display(Image(filename=str(result.mean_std_plot)))
if result.residual_plot:
    display(Image(filename=str(result.residual_plot)))

if result.validation_parity_plot:
    display(Image(filename=str(result.validation_parity_plot)))
if result.across_theta_plot:
    display(Image(filename=str(result.across_theta_plot)))
if result.across_theta_zoom_plot:
    display(Image(filename=str(result.across_theta_zoom_plot)))
if result.coverage_plot:
    display(Image(filename=str(result.coverage_plot)))

if result.theta_group_plot_dir:
    theta_plots = sorted(Path(result.theta_group_plot_dir).glob('*.png'))
    print('theta-group plots:', len(theta_plots), 'in', result.theta_group_plot_dir)
    for f in theta_plots:
        display(Image(filename=str(f)))
